In [ ]:
import os 

os.chdir("..")

In [ ]:
from jp_imports import JPTrade
from datetime import datetime
import polars as pl

jt = JPTrade()

In [ ]:
df = jt.process_int_jp(
            time_frame="qtr",
            level="hts",
            agriculture_filter=True,
            source="org",
            corrections=True,
        )
df = df.with_columns(
    hs4=pl.col("hts_code").str.slice(0, 4),
    imports_qty=pl.when(pl.col("imports_qty") == 0)
    .then(1)
    .otherwise(pl.col("imports_qty")),
    exports_qty=pl.when(pl.col("exports_qty") == 0)
    .then(1)
    .otherwise(pl.col("exports_qty")),
)
df = df.group_by(pl.col("year", "qtr", "hs4")).agg(
    imports=pl.col("imports").sum(),
    exports=pl.col("exports").sum(),
    imports_qty=pl.col("imports_qty").sum(),
    exports_qty=pl.col("exports_qty").sum(),
)
df = df.with_columns(
    price_imports=pl.col("imports") / pl.col("imports_qty"),
    price_exports=pl.col("exports") / pl.col("exports_qty"),
    date=pl.datetime(
            pl.col("year"), 
            (pl.col("qtr") - 1) * 3 + 1, 
            1
        ),
).sort(["hs4", "date"]).with_columns(
        # Calculate Year-over-Year (YoY) growth (lag of 4 quarters)
        price_imports_yoy=pl.col("price_imports").pct_change(4).over("hs4") * 100,
        price_exports_yoy=pl.col("price_exports").pct_change(4).over("hs4") * 100,
    )
df

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl

# 1. Filter for the latest date & prepare columns
latest_data = (
    df.filter(pl.col("date") == pl.col("date").max())
    .with_columns(
        actual_yoy=pl.col("price_imports_yoy"),
        plot_yoy=pl.col("price_imports_yoy").clip(-100, 100)
    )
)

# 2. Extract Top 50 and Bottom 50, sorted correctly
top_50_data = (
    latest_data.sort("actual_yoy", descending=True).head(50).to_pandas().iloc[::-1]
)
bottom_50_data = (
    latest_data.sort("actual_yoy", descending=False).head(50).to_pandas().iloc[::-1]
)

# 3. Create subplots with a professional layout structure
fig = make_subplots(
    rows=1, 
    cols=2, 
    subplot_titles=("Bottom 50 Categories (YoY % Decrease)", "Top 50 Categories (YoY % Increase)"),
    horizontal_spacing=0.06
)

# Shared hover template for clean, professional inspection
hover_template = (
    "<b>HS4 Code:</b> %{y}<br>"
    "<b>Price Imports YoY:</b> %{customdata:.1f}%<br>"
    "<extra></extra>"
)

# 4. Add Horizontal Bar Chart for Bottom 50 (Left Column) with INVERTED Viridis Palette ("Viridis_r")
fig.add_trace(
    go.Bar(
        x=bottom_50_data["plot_yoy"],
        y=bottom_50_data["hs4"],
        orientation="h",
        customdata=bottom_50_data["actual_yoy"],
        hovertemplate=hover_template,
        marker=dict(
            color=bottom_50_data["plot_yoy"],
            colorscale="Viridis_r",  # Inverted Viridis
            showscale=False,
            opacity=0.9,
            line=dict(color="rgba(0,0,0,0.1)", width=0.5),
        ),
        showlegend=False,
    ),
    row=1, col=1
)

# 5. Add Horizontal Bar Chart for Top 50 (Right Column) with Standard Viridis Palette
fig.add_trace(
    go.Bar(
        x=top_50_data["plot_yoy"],
        y=top_50_data["hs4"],
        orientation="h",
        customdata=top_50_data["actual_yoy"],
        hovertemplate=hover_template,
        marker=dict(
            color=top_50_data["plot_yoy"],
            colorscale="Viridis",  # Standard Viridis
            showscale=False,
            opacity=0.9,
            line=dict(color="rgba(0,0,0,0.1)", width=0.5),
        ),
        showlegend=False,
    ),
    row=1, col=2
)

# 6. Production-Ready Layout Adjustments (Fixed Title/Subtitle Overlap)
fig.update_layout(
    template="plotly_white",
    title=dict(
        text="<b>Import Price Volatility: Extreme YoY Changes</b><br><sup>Comparing the 50 largest positive and negative HS4 category movements</sup>",
        font=dict(size=18, family="Inter, sans-serif", color="#2C3E50"),
        x=0.5,
        xanchor="center",
        y=0.98,
        yanchor="top"
    ),
    font=dict(family="Inter, sans-serif", size=11, color="#34495E"),
    height=2200,
    margin=dict(l=80, r=80, t=160, b=80), 
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)"
)

# Configure X-Axes with clean grid lines
fig.update_xaxes(
    title_text="YoY % Change", 
    range=[-105, 0], 
    row=1, col=1,
    showgrid=True,
    gridcolor="#E5E8E8",
    zeroline=True,
    zerolinecolor="#2C3E50",
    zerolinewidth=1.5
)

fig.update_xaxes(
    title_text="YoY % Change", 
    range=[0, 105], 
    row=1, col=2,
    showgrid=True,
    gridcolor="#E5E8E8",
    zeroline=True,
    zerolinecolor="#2C3E50",
    zerolinewidth=1.5
)

# Clean Y-Axes styling
fig.update_yaxes(
    showticklabels=True, 
    tickfont=dict(size=9),
    showgrid=False,
    row=1, col=1
)
fig.update_yaxes(
    showticklabels=True, 
    tickfont=dict(size=9),
    showgrid=False,
    row=1, col=2
)

fig.show()